In [ ]:
library(ArchR)
library(data.table)
library(tidyverse)

addArchRThreads(threads = 8)
addArchRGenome("hg38")

pathToMacs2 <- "/data/leuven/345/vsc34531/miniconda/envs/archr_multiome/bin/macs2"
stopifnot(file.exists(pathToMacs2))

Sys.setenv(
  PATH = paste(dirname(pathToMacs2), Sys.getenv("PATH"), sep = .Platform$path.sep)
)

message("macs2 from Sys.which: ", Sys.which("macs2"))

In [ ]:
# Cell 2 — paths + exact sample sheet
proj_dir <- "/path/to/nature_study/ArchR"
frag_dir <- "/path/to/nature_study/fragments"
h5_dir   <- "/path/to/nature_study/h5"

In [ ]:
dir.create(proj_dir, recursive = TRUE, showWarnings = FALSE)

samples <- tibble::tribble(
  ~sample,       ~donor,     ~condition, ~fragments,                                                  ~h5,
  "H_5_A",       "H_5",      "disomy",   file.path(frag_dir, "H_5_A-atac_fragments.tsv.gz"),         file.path(h5_dir, "H_5_A-filtered_feature_bc_matrix.h5"),
  "H_5_B",       "H_5",      "disomy",   file.path(frag_dir, "H_5_B-atac_fragments.tsv.gz"),         file.path(h5_dir, "H_5_B-filtered_feature_bc_matrix.h5"),
  "H_6_A",       "H_6",      "disomy",   file.path(frag_dir, "H_6_A-atac_fragments.tsv.gz"),         file.path(h5_dir, "H_6_A-filtered_feature_bc_matrix.h5"),
  "H_7_A",       "H_7",      "disomy",   file.path(frag_dir, "H_7_A-atac_fragments.tsv.gz"),         file.path(h5_dir, "H_7_A-filtered_feature_bc_matrix.h5"),
  "H_7_B",       "H_7",      "disomy",   file.path(frag_dir, "H_7_B-atac_fragments.tsv.gz"),         file.path(h5_dir, "H_7_B-filtered_feature_bc_matrix.h5"),
  "H_7_C",       "H_7",      "disomy",   file.path(frag_dir, "H_7_C-atac_fragments.tsv.gz"),         file.path(h5_dir, "H_7_C-filtered_feature_bc_matrix.h5"),
  "TS21_4_H",    "TS21_4",   "Ts21",     file.path(frag_dir, "TS21_4_H-atac_fragments.tsv.gz"),      file.path(h5_dir, "TS21_4_H-filtered_feature_bc_matrix.h5"),
  "TS21_4_I",    "TS21_4",   "Ts21",     file.path(frag_dir, "TS21_4_I-atac_fragments.tsv.gz"),      file.path(h5_dir, "TS21_4_I-filtered_feature_bc_matrix.h5"),
  "TS21_11_F",   "TS21_11",  "Ts21",     file.path(frag_dir, "TS21_11_F-atac_fragments.tsv.gz"),     file.path(h5_dir, "TS21_11_F-filtered_feature_bc_matrix.h5"),
  "TS21_11_G",   "TS21_11",  "Ts21",     file.path(frag_dir, "TS21_11_G-atac_fragments.tsv.gz"),     file.path(h5_dir, "TS21_11_G-filtered_feature_bc_matrix.h5"),
  "TS21_16_A",   "TS21_16",  "Ts21",     file.path(frag_dir, "TS21_16_A-atac_fragments.tsv.gz"),     file.path(h5_dir, "TS21_16_A-filtered_feature_bc_matrix.h5"),
  "TS21_16_B",   "TS21_16",  "Ts21",     file.path(frag_dir, "TS21_16_B-atac_fragments.tsv.gz"),     file.path(h5_dir, "TS21_16_B-filtered_feature_bc_matrix.h5")
)

samples <- samples %>%
  mutate(
    fragment_index = paste0(fragments, ".tbi"),
    frag_exists    = file.exists(fragments),
    tbi_exists     = file.exists(fragment_index),
    h5_exists      = file.exists(h5)
  )

samples %>% select(sample, donor, condition, frag_exists, tbi_exists, h5_exists)
stopifnot(all(samples$frag_exists))
stopifnot(all(samples$tbi_exists))
stopifnot(all(samples$h5_exists))

write.csv(samples, file.path(proj_dir, "sample_sheet.csv"), row.names = FALSE)

In [ ]:
# Cell 3 — read filtered 10x barcodes from each H5
get_10x_barcodes <- function(h5_file) {
  rhdf5::h5read(h5_file, "matrix/barcodes")
}

validBarcodes <- setNames(
  lapply(samples$h5, get_10x_barcodes),
  samples$sample
)

summary(lengths(validBarcodes))
head(validBarcodes[[1]])

In [ ]:
# Cell 4 — create Arrow files
addArchRThreads(threads = 4)

arrow_dir <- file.path(proj_dir, "ArrowFiles")
dir.create(arrow_dir, recursive = TRUE, showWarnings = FALSE)

oldwd <- getwd()
setwd(arrow_dir)

ArrowFiles <- createArrowFiles(
  inputFiles      = setNames(samples$fragments, samples$sample),
  sampleNames     = samples$sample,
  validBarcodes   = validBarcodes,
  minTSS          = 4,
  minFrags        = 1000,
  addTileMat      = TRUE,
  addGeneScoreMat = TRUE,
  force           = FALSE
)

setwd(oldwd)

ArrowFiles

In [ ]:
samples <- samples %>%
  mutate(
    arrow = file.path(arrow_dir, paste0(sample, ".arrow")),
    arrow_exists = file.exists(arrow)
  )

samples %>% select(sample, arrow_exists)
stopifnot(all(samples$arrow_exists))

In [ ]:
# make a real QC output folder inside your project
qc_dir <- file.path(proj_dir, "QualityControl")
dir.create(qc_dir, recursive = TRUE, showWarnings = FALSE)

qc_dir
dir.exists(qc_dir)

In [ ]:
addArchRThreads(threads = 4)

ArrowFiles <- addDoubletScores(
  input     = ArrowFiles,
  k         = 10,
  knnMethod = "UMAP",
  LSIMethod = 1,
  outDir    = qc_dir,
  force     = TRUE
)

ArrowFiles

In [ ]:
# rebuild the ArrowFiles character vector
ArrowFiles <- file.path(arrow_dir, paste0(samples$sample, ".arrow"))
names(ArrowFiles) <- samples$sample

stopifnot(all(file.exists(ArrowFiles)))
ArrowFiles

# Cell 6 — create ArchR project
proj <- ArchRProject(
  ArrowFiles      = ArrowFiles,
  outputDirectory = file.path(proj_dir, "ArchRProject"),
  copyArrows      = FALSE
)

proj <- filterDoublets(proj)
proj

In [ ]:
# Cell 7 — add donor and condition metadata
cell_md <- getCellColData(proj, select = "Sample") %>%
  as.data.frame() %>%
  tibble::rownames_to_column("cellNames") %>%
  left_join(
    samples %>% select(sample, donor, condition),
    by = c("Sample" = "sample")
  )

stopifnot(!any(is.na(cell_md$donor)))
stopifnot(!any(is.na(cell_md$condition)))

proj <- addCellColData(
  ArchRProj = proj,
  data      = cell_md$donor,
  cells     = cell_md$cellNames,
  name      = "donor",
  force     = TRUE
)

proj <- addCellColData(
  ArchRProj = proj,
  data      = cell_md$condition,
  cells     = cell_md$cellNames,
  name      = "condition",
  force     = TRUE
)

table(proj$donor, proj$condition)


In [ ]:
# 8) Save project
# ------------------------------------------------------------
proj <- saveArchRProject(
  ArchRProj = proj,
  outputDirectory = file.path(proj_dir, "ArchRProject_saved"),
  load = TRUE
)

In [ ]:
# 9) Import RNA annotations
# ------------------------------------------------------------
rna_anno <- read.delim("/path/to/nature_study/ArchR/RNA_annotations/rna_annotations_for_ArchR.tsv")

common <- intersect(getCellNames(proj), rna_anno$cellNames)
rna_anno <- rna_anno[match(common, rna_anno$cellNames), , drop = FALSE]

proj <- addCellColData(proj, data = rna_anno$rna_celltype, cells = rna_anno$cellNames, name = "rna_celltype", force = TRUE)
proj <- addCellColData(proj, data = rna_anno$stem_core,   cells = rna_anno$cellNames, name = "stem_core",   force = TRUE)